# Example queries: `annual` (comstock_oedi)

Auto-generated from `tests/query_snapshots/annual.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `annual_totals_overall`

Annual electricity + natural gas totals restricted to CO.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh', 'out.natural_gas.total.energy_consumption..kwh'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


## `annual_totals_by_building_type`

Annual totals grouped by building type, CO only. Two variants confirm that omitting annual_only (default True) matches passing annual_only=True explicitly.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh', 'out.natural_gas.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


## `annual_electricity_by_vintage_sort`

Annual electricity grouped by vintage with sort, limit 5, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['vintage'],
    restrict=[('state', ['CO'])],
    sort=True,
    limit=5,
)
result.head() if hasattr(result, 'head') else result


## `annual_electricity_by_vintage_nosort`

Annual electricity by vintage with sort=false + limit. Hits the no-ORDER-BY branch which existing entries never exercise. Marked nondeterministic because LIMIT without ORDER BY makes Trino free to return any 5 rows; only SQL shape is meaningful.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['vintage'],
    restrict=[('state', ['CO'])],
    sort=False,
    limit=5,
)
result.head() if hasattr(result, 'head') else result


## `annual_nonzero_count_natural_gas`

Annual natural gas with nonzero_units_count requested, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.natural_gas.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    get_nonzero_count=True,
)
result.head() if hasattr(result, 'head') else result


## `annual_baseline_quartiles`

Annual baseline (upgrade=0) electricity with get_quartiles=true. Hits the quartile branch on the upgrade-0 path where there is no upgrade-table join — the existing quartile coverage is upgrade=1 only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    get_quartiles=True,
)
result.head() if hasattr(result, 'head') else result


## `annual_agg_func_mean`

Annual electricity with agg_func='mean'. Pins the non-sum aggregation branch (no weight multiplication, AVG instead of SUM).


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    agg_func='mean',
)
result.head() if hasattr(result, 'head') else result


## `annual_agg_func_max`

Annual electricity with agg_func='max'. Pins the MAX() aggregation branch.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    agg_func='max',
)
result.head() if hasattr(result, 'head') else result


## `annual_agg_func_min`

Annual electricity with agg_func='min'. Pins the MIN() aggregation branch.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    agg_func='min',
)
result.head() if hasattr(result, 'head') else result


## `annual_with_extra_weight_sqft`

Annual electricity with weights=[$SQFT_COL] — resstock 'in.sqft', comstock 'in.sqft..ft2'. Pins the SQL shape where the user supplies an explicit additional weight column on top of the default sample_weight; Athena gets `electricity * sample_weight * sqft`. Internally exercised by utility queries (eiaid_weights.weight); no other entry covers user-supplied weights directly.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    weights=['in.sqft..ft2'],
)
result.head() if hasattr(result, 'head') else result


## `annual_agg_func_arbitrary`

Annual electricity with agg_func='arbitrary'. Pins the ARBITRARY() (Athena's any-value) aggregation branch. Marked nondeterministic because arbitrary() picks any group value per row; only SQL shape is meaningful.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    agg_func='arbitrary',
)
result.head() if hasattr(result, 'head') else result
